# Step 6: Evaluation — Hold-outs + Forecast Skill

## What this notebook checks

After training, we need honest numbers — not just “looks good on shuffled training data”.

| Check | Question it answers |
|-------|---------------------|
| **Time hold-out** | Can the model predict **future** dates it never trained on? |
| **Location hold-out** | Does it generalise to a reef site left out of training? |
| **Forecast skill** | Is a 1 / 3 / 7-day PINN forecast better than simple baselines? |

**Baselines we beat (or fail to beat):**
- **Persistence** — “tomorrow = today” (last known SST)
- **Climatology** — monthly average for that location

## Prerequisites

Run these first:
1. `04_real_spatial_data.ipynb`
2. `05_estimate_advection.ipynb` (recommended)
3. `02_pinn_model.ipynb` (re-train so the model matches the new scalers)

## Import libraries

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from evaluate_holdout import run as run_holdout
from validate_forecast import run as run_forecast_validation

print("TensorFlow", tf.__version__)
print("Looking for model:", os.path.exists("pinn_model_best.h5"))

## Part A — Hold-out evaluation

Reports MAE / RMSE in **°C** on:
- validation dates (recent, unseen)
- test dates (newest, unseen)
- location hold-out (default: Trincomalee)

In [ ]:
holdout_results = run_holdout()

### How to read Part A

- **MAE ≈ 0.5–1.0°C** → usable for a prototype
- **Location hold-out much worse than time test** → model memorised site quirks, weak spatial physics
- **Test ≫ Val** → possible distribution shift / need more training data

## Part B — Forecast skill (1, 3, 7 days)

For each horizon we compare three predictors on the **time test split**:

```
PINN          → model(lat, lon, future_time)
Persistence   → SST from h days ago
Climatology   → monthly mean SST for that reef
```

**Skill score** (positive = PINN wins):

$$
\text{skill} = 1 - \frac{\text{MAE}_{\text{PINN}}}{\text{MAE}_{\text{baseline}}}
$$

In [ ]:
forecast_out = run_forecast_validation()
summary = forecast_out["summary"]
detail = forecast_out["detail"]
summary

### Per-location detail

Useful to see if one reef is dragging the average down.

In [ ]:
if detail is not None and len(detail):
    display_cols = [
        "location", "horizon_days", "n",
        "pinn_mae", "persistence_mae", "climatology_mae",
    ]
    detail[display_cols].sort_values(["horizon_days", "location"])
else:
    print("No detail rows — did you run prepare_data + retrain?")

## Plot MAE by horizon

In [ ]:
if summary is not None and len(summary):
    fig, ax = plt.subplots(figsize=(8, 5))
    x = summary["horizon_days"].values
    width = 0.25
    ax.bar(x - width, summary["pinn_mae"], width, label="PINN")
    ax.bar(x, summary["persistence_mae"], width, label="Persistence")
    ax.bar(x + width, summary["climatology_mae"], width, label="Climatology")
    ax.set_xticks(x)
    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel("MAE (°C)")
    ax.set_title("Forecast skill: lower is better", fontweight="bold")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nSkill vs persistence (positive = PINN better):")
    for _, row in summary.iterrows():
        print(f"  {int(row['horizon_days'])}-day: {row['skill_vs_persistence']:+.3f}")
else:
    print("No summary to plot.")

## Part C — Quick DHW risk sanity check

Bleaching risk now leans on **Degree Heating Weeks** (NOAA-style):
- DHW ≥ 4 → warning
- DHW ≥ 8 → danger / mass bleaching risk

In [ ]:
from utils import calculate_bleaching_risk

examples = [
    {"label": "Cool / no DHW", "temp": 27.5, "base": 28.0, "dhw": 0.0},
    {"label": "Warm, DHW=4", "temp": 30.0, "base": 28.0, "dhw": 4.0},
    {"label": "Hot, DHW=8", "temp": 31.5, "base": 28.0, "dhw": 8.0},
]

rows = []
for ex in examples:
    r = calculate_bleaching_risk(
        current_temp=ex["temp"],
        baseline_temp=ex["base"],
        recent_temps=[ex["temp"]] * 7,
        dhw=ex["dhw"],
    )
    level = {0: "Healthy", 1: "Warning", 2: "Danger"}[r["risk_level"]]
    rows.append({
        "case": ex["label"],
        "temp": ex["temp"],
        "dhw": ex["dhw"],
        "risk_score": round(r["risk_score"], 3),
        "risk_level": level,
    })

pd.DataFrame(rows)

## Summary checklist

- [ ] Time test MAE looks reasonable (°C)
- [ ] Location hold-out is not a disaster
- [ ] PINN skill vs persistence is ≥ 0 at short horizons (or you know why not)
- [ ] DHW examples map to Healthy / Warning / Danger as expected

If skill is negative, the PINN is **not** yet a better forecaster than “use yesterday’s SST” — keep iterating on physics weight, advection, and training length.

**Outputs written:**
- `holdout_evaluation.pkl`
- `forecast_validation.csv`
- `forecast_validation_summary.csv`